# Interactive plots

Every plot in ocean-skill is described as a backend-agnostic **spec** and drawn by a
**renderer**. Choosing a different renderer is the only change needed to go from a static
figure to an interactive one — the comparison, the alignment and the metrics are
identical.

```python
c.plot()                       # matplotlib (default)
c.plot(renderer="holoviews")   # interactive
c.plot(renderer="both")        # side by side
```

See `getting_started.ipynb` for the comparison workflow itself; this notebook is only
about the rendering.

> **Interactive output needs a live notebook.** On GitHub or in a static export the cells
> below may appear blank — run the notebook, or export with
> `holoviews.save(obj, "plot.html")`, which writes a self-contained HTML file with
> BokehJS embedded.

## Setup

The bokeh extension has to be active for holoviews output to display inline.

In [ ]:
import holoviews as hv
import panel as pn

import ocean_skill as osk

hv.extension("bokeh")
pn.extension()

TEMP = "sea_water_potential_temperature"
SALT = "sea_water_practical_salinity"

In [ ]:
depths = osk.compare(
    aggregate={"time": "mean"},
    reference="woa23_temperature_month01",
    test="roms_example_combined",
    variables=[TEMP],
    depths=(0, 50, 100, 300),
)
depths

## 1. An interactive field row

The same `test | reference | difference` row, drawn with holoviews. The three panels share
pan and zoom, and hovering a cell reads out its value and position — which a static figure
cannot do.

In [ ]:
depths[0].plot(renderer="holoviews", title="Surface temperature")

## 2. An interactive Target diagram

This is where interactivity earns the most: a summary diagram compresses each comparison
to a single point, so the numbers behind it have nowhere to go. Hover a point to get its
**whole metric record** — bias, RMSE, correlation, σ-ratio and sample count.

In [ ]:
depths.target(renderer="holoviews", title="Temperature vs depth")

## 3. Static and interactive side by side

`renderer="both"` returns a panel `Row`: the matplotlib figure on the left, the live
holoviews object on the right. matplotlib and bokeh cannot share a figure, so panel hosts
each in its own pane.

Useful for checking that the two agree — and for handing colleagues a page where the
static version is the record and the interactive one is for exploring.

In [ ]:
depths.target(renderer="both")

In [ ]:
depths[0].plot(renderer="both", title="Surface temperature")

## 4. Several variables

Grouping works the same in both backends. Here two variables, each with its own row.

In [ ]:
physics = osk.compare(
    aggregate={"time": "mean"},
    reference=["woa23_temperature_month01", "woa23_salinity_month01"],
    test="roms_example_combined",
    variables=[TEMP, SALT],
)
physics.plot(renderer="holoviews")

## 5. What is *not* interactive, and why

**Taylor diagrams stay static.** A Taylor diagram is drawn on a floating polar axis
(`mpl_toolkits.axisartist`) with a curved correlation arc — bokeh has no equivalent, and
rebuilding it from primitives is a project rather than a port. Asking for an interactive
Taylor therefore returns the matplotlib figure and warns, rather than failing:

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fig = depths.taylor(renderer="holoviews")
    print([str(w.message)[:90] for w in caught][:1])
print("returned:", type(fig).__name__)

`paired` (Taylor + Target together) delegates for the same reason. If you want an
interactive summary, use `target(renderer="holoviews")` and show Taylor beside it
statically.

## 6. Saving

holoviews objects export to a standalone HTML file with BokehJS embedded — no server, no
notebook needed to view it:

```python
obj = depths.target(renderer="holoviews")
hv.save(obj, "target.html")            # ~25 KB

both = depths.target(renderer="both")  # a panel object
both.save("target_both.html", embed=True)
```

Maps are considerably larger (a few MB) because every cell value is embedded. For big
domains consider `rasterize=True` in a custom call, or export a static figure instead.

## Where this is heading

Deliberately **not** decided yet: where interactive output ultimately lives — inline in
notebooks, published into a Jupyter Book, or served as a panel dashboard. Each implies a
different export path, so the current target is simply notebook-inline plus standalone
HTML, which works for all three.

Animations (time sliders, mp4/gif export) are the other open piece, and depend on the
same decision.